In [1]:
import sys
from all_datasets.COOS_dataset import COOSDataset
from all_datasets.FMoW_dataset import FMoWDataset
from all_datasets.CivilComments_dataset import CivilCommentsDataset
import numpy as np
import matplotlib.pyplot as plt
import h5py
from get_clip_embeddings import get_text_embeddings, get_dataloader, preprocess_images, preprocess_texts, fix_embedding
import torch

/data/vision/beery/scratch/neha/.local/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/data/vision/beery/scratch/neha/.local/lib/python3.10/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


In [ ]:
from transformers import CLIPProcessor, CLIPModel
dataloader = get_dataloader("CivilComments","train")
model_name = "openai/clip-vit-base-patch32"
clip_processor = CLIPProcessor.from_pretrained(model_name)
clip_model = CLIPModel.from_pretrained(model_name)

def preprocess_texts(texts, clip_processor, max_length):
    # Split texts into chunks of max_length
    inputs = []
    for text in texts:
        tokens = clip_processor.tokenizer.tokenize(text)
        token_chunks = [tokens[i:i+max_length-2] for i in range(0, len(tokens), max_length-2)]
        # token_chunks = [tokens[i:i+max_length] for i in range(0, len(tokens), max_length)]
        chunked_texts = [clip_processor.tokenizer.convert_tokens_to_string(chunk) for chunk in token_chunks]
        inputs.append(chunked_texts)
    return inputs

def get_embeddings_1(data_loader, clip_processor, clip_model):
    embeddings_list = []
    max_length = clip_processor.tokenizer.model_max_length

    for images, texts, labels, uids in data_loader:
        text_chunks = preprocess_texts(texts, clip_processor, max_length)
        
        for chunks, text, uid in zip(text_chunks, texts, uids):
            chunk_embeddings = []
            
            for chunk in chunks:
                text_inputs = clip_processor(text=[chunk], padding=True, truncation=True, return_tensors='pt')
                print(len(text_inputs))
                with torch.no_grad():
                    text_features = clip_model.get_text_features(**text_inputs)
                    text_embeddings = text_features / text_features.norm(dim=-1, keepdim=True)
                    chunk_embeddings.append(text_embeddings)
            
            # Ensure chunk_embeddings have the same dimensions before stacking
            chunk_embeddings = torch.stack([emb.squeeze(0) for emb in chunk_embeddings])
            
            # Aggregate chunk embeddings (e.g., average)
            aggregated_embedding = torch.mean(chunk_embeddings, dim=0)
            
            embeddings_list.append({
                "text_embedding": aggregated_embedding.numpy(), 
                "text": text, 
                "uid": uid
            })
            
    return embeddings_list
    
def fix_embedding(embedding):
    dict_embed = {}
    for e in embedding:
        for key, value in e.items():
            if key not in dict_embed:
                dict_embed[key] = []
            dict_embed[key].append(value)
    return dict_embed

embedding=get_embeddings_1(dataloader, clip_processor, clip_model)

Token indices sequence length is longer than the specified maximum sequence length for this model (104 > 77). Running this sequence through the model will result in indexing errors


2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2


In [ ]:
file_path = '/data/vision/beery/scratch/neha/task-datacomp/all_datasets/COOS/COOS7_train.hdf5'
data_file = h5py.File(file_path, 'r')
# print(type(data_file['data'][:]))
total_samples = len(data_file['data'])
min_val = np.max(data_file['data'][:])
print(min_val)

In [ ]:
dataset=COOSDataset('train')
img, text, label, uid = dataset[89]
plt.imshow(img.permute(1,2,0))

In [ ]:
#### embed=np.load('/data/vision/beery/scratch/neha/task-datacomp/all_datasets/embeddings/COOS_train_normalemebed.npy',allow_pickle=True)

###### print(embed[0])

In [ ]:
embedding = {}
for e in embed:
    for key, value in e.items():
        if key not in embedding:
            embedding[key] = []
        embedding[key].append(value)

In [ ]:
len(embedding['image_embedding'])

In [ ]:
embedding.keys()

In [ ]:
centroids=np.load('/data/vision/beery/scratch/neha/task-datacomp/all_datasets/embeddings/centroids.npy')
centroids

In [ ]:
embed = [e["image_embedding"] for e in embed]
embed[0].shape

In [ ]:
np.vstack(embed).shape

In [ ]:
dataset=FMoWDataset('train')
img, text, label, uid = dataset[23]
plt.imshow(img.permute(1,2,0))

In [ ]:
embedding_new=fix_embedding(embedding)

In [ ]:
len(embedding_new['image_embedding'])